# Bronze - ATP Ranking

## Imports

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f

import pandas as pd

import os
os.environ['SPARK_LOCAL_IP'] = '127.0.0.1'

from dotenv import load_dotenv
load_dotenv()

True

## Init spark

In [2]:
try:
    spark = SparkSession.builder.appName("bronze_ranking").getOrCreate()
except Exception as e:
    print(e)

## Load database

In [9]:

tb_ranking_historical = spark.read.format("csv").option("header", "true").load(r"../../data/raw/historical/tb_ranking_historical.csv")
tb_ranking_weekly = spark.read.format("csv").option("header", "true").load(r"../../data/raw/incremental/tb_ranking_weekly.csv")

## Append new data

In [18]:
new_rankings = (
    tb_ranking_historical.alias("h")
    .join(
        tb_ranking_weekly.alias("w"),
        [
            f.col("h.date_week") == f.col("w.date_week"), 
            f.col("h.player") == f.col("w.player")
        ],
        'right'
        )
    .where(f.col("h.date_week").isNull())
    .select(
        "w.*",
    )
)

In [20]:
df = tb_ranking_historical.unionByName(new_rankings, allowMissingColumns=True).withColumn("DATE_INGESTION", f.lit(f.current_date()))

## Save dataframe

### Local

In [21]:
df.toPandas().to_csv(
    r"../../data/bronze/tb_atp_rankings.csv",
    index=False,
    sep=",",
    encoding="utf-8"
)

### Supabase

In [22]:
(
df.write
    .format("jdbc")
    .option("url", os.getenv("JDBC_URL"))
    .option("dbtable", "bronze.tb_atp_rankings")
    .option("user", os.getenv("DB_USER"))
    .option("password", os.getenv("DB_PASSWORD"))
    .option("driver", "org.postgresql.Driver")
    .mode("overwrite")
    .save()
)